In [ ]:
# =============================================================================
# 1. Setup
# Ocean heat content (OHC) forward emulator for ACCESS-OM2.
#
# This notebook only holds the experiment settings (cell 2) and the calls that
# run it. The data pipeline, model, losses, training loop and diagnostics live
# in src/Experiment/ (built on src/Emulator/ for the UNet and the loss terms).
#
# Cells: 1 setup | 2 config | 3 data | 4 known-closure check | 5 model + losses
#        6 training | 7 skill test | 8 control run | 9 plots
# =============================================================================
import os
import sys

# Let the CUDA allocator grow segments instead of fragmenting memory. Must be set
# before the first CUDA call, so restart the kernel for it to take effect.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Path to the OM2-emulator repository. Leave as None to find it automatically;
# set it (e.g. "/home/156/txs156/uom/OM2-emulator") if that fails.
REPO_ROOT = "/home/156/txs156/uom/OM2-emulator"

# Places to look for src/Experiment, in order: REPO_ROOT, the OM2_EMULATOR_REPO
# environment variable, the notebook file's folder (VS Code provides it as
# __vsc_ipynb_file__), then the working directory. Each is searched upwards,
# because the kernel's working directory is not always the notebook's folder.
_starts = [
    Path(p).expanduser()
    for p in (REPO_ROOT, os.environ.get("OM2_EMULATOR_REPO"), globals().get("__vsc_ipynb_file__"), Path.cwd())
    if p
]
_candidates = [q for p in _starts for q in (p, *p.parents)]
SRC = next((q / "src" for q in _candidates if (q / "src" / "Experiment").is_dir()), None)
if SRC is None:
    raise FileNotFoundError(
        f"Could not find src/Experiment. Working directory: {Path.cwd()}. Looked in and above: "
        f"{[str(p) for p in _starts]}. Set REPO_ROOT above to the OM2-emulator repository."
    )
sys.path.insert(0, str(SRC))

from Experiment import (
    ClosureConfig, DataConfig, EvalConfig, ExperimentConfig, LossConfig, ModelConfig, NormalisationConfig,
    TimeConfig, TrainConfig, WindowConfig,
    build_data, build_losses, build_model, build_module, build_trainer,
    check_known_closure, count_parameters, run_control, run_skill_test,
    plot_control, plot_global_rmse_all_variables, plot_rmse_by_epoch, plot_skill_evaluation,
    leading_growth_mode, persistence_rmse, plot_growth_mode,
    plot_training_history, plot_skill_diagnostics, plot_spectra, skill_summary,
)

print(f"src: {SRC}")
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [ ]:
# =============================================================================
# 2. Experiment configuration -- every setting of the run lives here.
# Dates are "YYYY-MM" and all date ranges are inclusive. The config is checked
# for consistency as soon as it is created.
# =============================================================================
cfg = ExperimentConfig(
    # ---- Variables and normalisation ---------------------------------------
    data=DataConfig(
        path="/g/data/nm47/txs156/OM2-emulator/data/1deg_ocean_heat_emulator_data.nc",
        # Prognostic = predicted AND fed back as inputs at the next step (the outputs).
        prognostic=["ocean_heat_content_2d", "ocean_freshwater_content_2d", "eta_t", "surface_speed"],
        # Forcing = prescribed from the data at every step. Drop tau_x/tau_y for
        # heat-flux-only forcing.
        forcing=["total_surface_heat_flx", "ocean_freshwater_flux", "tau_x", "tau_y"],
        # Land boundary condition per variable (default Neumann: land takes the nearest
        # ocean value, i.e. no flux across the coast). Add velocity components as
        # Dirichlet (land held at z = 0, no normal flow), e.g. {"u": "dirichlet"}.
        # See "Land boundary conditions" in src/Emulator/om2_model_utils.py.
        boundary_conditions={},
        normalisation=NormalisationConfig(
            # Spatial_climatology | Spatial_climatology_global_variance | Spatial_in_time | Global_in_time
            strategy="Spatial_climatology",
            fit_period=("1970-01", "2004-02"),  # months used to fit mean/std (inside the training period)
        ),
        cache_dir=None,  # None -> <data dir>/emulator_cache; rebuilt automatically when data settings change
    ),
    # ---- Time periods --------------------------------------------------------
    # train/valid are the INITIAL months of the samples (the last prior state);
    # each sample also needs n_prior - 1 earlier and posterior_steps later months.
    time=TimeConfig(
        time_axis=("1970-01", "2018-12"),
        train=("1970-02", "2004-01"),
        valid=("2004-02", "2005-01"),
        test=("2005-02", "2018-11"),     # skill test: seeded at 2005-02, predicts 2005-03 .. 2018-12
        control=("2003-01", "2003-12"),  # control run: this year's forcing repeated (near-zero net flux)
        control_repeats=10,
    ),
    # ---- Rollout windows -----------------------------------------------------
    window=WindowConfig(
        n_prior=2,             # past states the model sees
        posterior_steps=36,    # months per sample window (free-running + scored steps)
        rollout_steps=12,      # autoregressive steps per training sample, or [n_free, n_trained]
        # Curriculum {first_epoch: steps}: cost scales with the trained steps, so early
        # epochs are cheap and the model learns one-step dynamics first. [24, 12] =
        # "pushforward": 24 steps without gradients from the true state, then the loss
        # on the next 12, so the model trains on its own 2-3-year-old states, where the
        # long-rollout damping and noise appear. None = always rollout_steps.
        rollout_schedule={0: 1, 5: 4, 10: 12, 15: [24, 12]},
        valid_rollout_steps=36,  # validation scores all 36 months (None -> all of rollout_steps)
    ),
    # ---- Loss terms and weights ----------------------------------------------
    loss=LossConfig(
        # local_mse | spectral | <budget>_closure for each budget in `closures`.
        # 0 switches a term off. A weight can also be a list of absolute per-step
        # weights (length posterior_steps).
        terms={"local_mse": 1.0, "spectral": 2.0, "heat_closure": 0.1, "freshwater_closure": 0.1},
        step_weights=None,     # relative emphasis per lead time, e.g. [1] * 6 + [2] * 6 (rescaled to mean 1)
        # Global budgets: d/dt sum(area * content) = sum(area * flux) on anomalies.
        # The flux must be in content units per second; surface_flux_sign=+1 means
        # positive flux enters the ocean; min_scale floors the residual's scale.
        closures={
            "heat": ClosureConfig(
                content_variable="ocean_heat_content_2d", flux_variable="total_surface_heat_flx",
                surface_flux_sign=1.0, min_scale=1.0e20,  # J
            ),
            "freshwater": ClosureConfig(
                content_variable="ocean_freshwater_content_2d", flux_variable="ocean_freshwater_flux",
                surface_flux_sign=1.0, min_scale=1.0e12,  # kg
            ),
        },
    ),
    # ---- Architecture ----------------------------------------------------------
    model=ModelConfig(
        arch="unet",
        padding_mode="zeros",  # south edge only; longitude is periodic and the north edge is the tripole fold
        # Channel widths = width_multiplier x input channels (n_prior * prognostic + forcing),
        # doubling with depth. The UNet predicts the change from the last state and
        # gets the raw input at full resolution. Memory grows ~linearly with this.
        width_multiplier=4,
        channels_last=True,    # NHWC layout: faster fp16 convolutions on the V100's tensor cores
        # "spatial_residual" = the former ResNet notebook: a learned full-resolution
        # correction of the prediction. Adds memory: lower batch_size if needed.
        output_head=None,
        output_head_hidden_channels=16,
        output_head_residual_scale=1.0,
        latent_processor=None,  # or "latent_residual" (refines the UNet bottleneck)
    ),
    # ---- Training ---------------------------------------------------------------
    train=TrainConfig(
        batch_size=64,
        accumulate_grad_batches=1,  # effective batch = batch_size * this
        max_epochs=20,               # 10 warm-up epochs, 5 at 12 steps, 5 at [24 free, 12 trained]
        lr=1e-4,
        weight_decay=0.0,
        lr_scheduler=None,          # or "cosine"
        gradient_clip_val=None,
        precision="16-mixed",       # fp16 on the V100's tensor cores: ~1.5-2x faster, half the activations
        # Recompute each rollout step in the backward pass: ~1/rollout_steps of the
        # activation memory for ~30% more compute. Turn off if the batch fits without it.
        checkpoint_rollout_steps=True,
        # Keep this many steps' activations (no recompute) and checkpoint the rest:
        # 2 fits at 12 steps with batch 64 on a 32 GB V100 (4 did not).
        rollout_steps_in_memory=2,
        compile_model=False,        # torch.compile: experimental; slow first steps while compiling
        seed=42,
        run_dir="../runs/pushforward_spectral",  # saves config, CSV log (per-term losses), checkpoints
    ),
    # ---- Evaluation -------------------------------------------------------------
    eval=EvalConfig(
        run_skill_test=True,
        run_control=False,
        snapshot_periods=["2005-05", "2010-12", "2015-12"],  # months "YYYY-MM" or year means "YYYY"
        # Map colour limits: None = automatic per variable (99th percentile of |field|),
        # or {variable: limit} to fix some of them.
        anomaly_scale=None,
        difference_scale=None,
    ),
)

In [ ]:
# =============================================================================
# 3. Data: normalise every variable once over the whole time axis (cached on
# disk, keyed by the data settings), then serve rollout windows from it.
# =============================================================================
data = build_data(cfg)

In [ ]:
# =============================================================================
# 4. Known-closure check: each active closure loss on the TRUE data, i.e. the
# lowest value that <budget>_closure term can reach. See issue #43: with monthly
# means the anomaly heat budget scores ~0.5, so a perfect model is still penalised.
# =============================================================================
known_closure = check_known_closure(cfg, data) if cfg.active_closures() else None

In [ ]:
# =============================================================================
# 5. Model, loss terms and the Lightning module.
# =============================================================================
model = build_model(cfg, data)
losses = build_losses(cfg, data)
module = build_module(cfg, model, losses, data)

print(
    f"Model: {cfg.model.arch} | output_head={cfg.model.output_head} | "
    f"latent_processor={cfg.model.latent_processor} | {count_parameters(model):,} parameters"
)
for term in losses:
    print(f"  {term.name:15s} step weights {term.step_weights}")

In [ ]:
%%time
# =============================================================================
# 6. Training. Check nvidia-smi first: other kernels on the same GPU slow
# training down and can cause CUDA out-of-memory errors (build_trainer warns).
# =============================================================================
trainer = build_trainer(cfg)
trainer.fit(module, data.train_dl, data.valid_dl)

In [ ]:
# =============================================================================
# 7. Skill test: one free-running rollout from the truth at time.test[0],
# driven by the real (held-out) forcing. Returns predictions and truth in
# physical units, as full fields and as anomalies.
# =============================================================================
skill = run_skill_test(cfg, model, data) if cfg.eval.run_skill_test else None
skill

In [ ]:
# =============================================================================
# 7b. Stability diagnostic: the fastest-growing small perturbation of the
# one-step map around the true state at time.test[0]. A growth factor > 1 means
# rollouts amplify that pattern every step (e.g. 1.08 -> ~1e5x over 14 years).
# =============================================================================
growth_mode = leading_growth_mode(model, data, cfg.time.test[0])
print(f"Leading growth factor: {growth_mode['growth']:.4f} per step")


In [ ]:
# =============================================================================
# 8. Control run: the forcing of time.control repeated time.control_repeats
# times, to isolate drift under near-constant forcing.
# =============================================================================
control = run_control(cfg, model, data) if cfg.eval.run_control else None

In [ ]:
# =============================================================================
# 9. Plots, saved to FIG_DIR for every prognostic variable in cfg.data.prognostic:
#   <variable>_skill_evaluation.png : anomaly maps + global-integrated anomaly
#   allvars_RMSE_evaluation.png     : global area-weighted RMSE through the skill test, with persistence
#   allvars_RMSE_epoch.png          : train/validation RMSE per epoch, with validation persistence
# =============================================================================
FIG_DIR = SRC.parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")
    print(f"Saved {FIG_DIR / name}")

if skill is not None:
    for variable in cfg.data.prognostic:
        save(plot_skill_evaluation(skill, variable, cfg.eval.snapshot_periods,
                                   cfg.eval.anomaly_scale, cfg.eval.difference_scale),
             f"{variable}_skill_evaluation.png")
    save(plot_global_rmse_all_variables(skill, cfg.data.prognostic), "allvars_RMSE_evaluation.png")
save(plot_growth_mode(growth_mode, data), "growth_mode.png")
if not module.rmse_history.empty:
    # Persistence baseline on the validation windows, in the same units as the per-epoch RMSE.
    persistence = persistence_rmse(cfg, data)
    print("Validation persistence RMSE:", {k: round(v, 3) for k, v in persistence.items()})
    save(plot_rmse_by_epoch(module.rmse_history, cfg.data.prognostic, baseline=persistence), "allvars_RMSE_epoch.png")
if control is not None:
    plot_control(control)
if cfg.train.run_dir is not None:
    plot_training_history(cfg.train.run_dir)
plt.show()


In [ ]:
# =============================================================================
# 10. Skill-test diagnostics: is the error in the pattern, the amplitude or the scales?
#   - RMSE against persistence AND climatology (zero anomaly). Where nothing is
#     predictable, zero anomaly beats persistence by ~sqrt(2): beating
#     persistence alone is not skill.
#   - Anomaly correlation (ACC): the pattern match, whatever its amplitude.
#   - Anomaly amplitude: below the truth = fading (blurring), above = noise/drift.
#   - Zonal power spectra vs physical wavenumber (cycles/km), on the same bands
#     as the spectral loss, so they read the same at 1 deg and 1/10 deg.
# Table: first lead month at which ACC < 0.5 and the emulator is worse than
# persistence / climatology (None = never); emulator/truth amplitude, last year.
# =============================================================================
if skill is not None:
    display(skill_summary(skill, cfg.data.prognostic))
    save(plot_skill_diagnostics(skill, cfg.data.prognostic), "allvars_skill_diagnostics.png")
    save(plot_spectra(skill, cfg.data.prognostic), "allvars_spectra.png")